# Diagnóstico 05 — Organização física dos Parquets e leitura seletiva

## Objetivo

Investigar se a organização física dos arquivos pode reduzir o custo de leitura das bases `grupo` e `dsp`, sem alterar os dados analíticos consumidos pelo dashboard.

Este diagnóstico parte de uma hipótese, não de uma garantia: **o consumo pode ser inerente à quantidade e à natureza dos dados**. Os testes servem para descobrir se há margem de otimização na organização física.

Serão comparadas quatro alternativas:

1. **Original:** arquivo atual, com sua organização física existente.
2. **Ordenado + Row Groups de 50 mil linhas.**
3. **Ordenado + Row Groups de 100 mil linhas.**
4. **Ordenado + Row Groups de 250 mil linhas.**
5. **Particionado por `TIPO_REFERENCIA_TERRITORIAL`.**

Nas alternativas ordenadas, as linhas são ordenadas pela coluna usada no filtro antes da gravação. Isso pode tornar as estatísticas `min`/`max` dos Row Groups mais seletivas. Na alternativa particionada, cada categoria da referência territorial é gravada em uma partição própria.

**Importante:** todos os arquivos gerados ficam em uma pasta temporária de diagnóstico. O notebook não sobrescreve os Parquets originais nem modifica o pipeline do projeto.


## 1. Configuração e caminhos

Execute o notebook a partir da raiz de `app_render`, como nos diagnósticos anteriores. A configuração procura `dash_utils/config.py` e utiliza o diretório de consumo definido pelo projeto.

Antes de executar a geração das alternativas, confira o espaço livre em disco. Serão criadas cópias temporárias das duas bases, com as colunas utilizadas pelo dashboard.


In [1]:
from pathlib import Path
import os
import sys
import json
import time
import gc
import shutil
import subprocess
import importlib
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dash_utils" / "config.py").exists():
    candidatos = [
        Path.cwd().parent,
        Path.cwd() / "app_render",
        Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render",
    ]
    PROJECT_ROOT = next(
        (p.resolve() for p in candidatos if (p / "dash_utils" / "config.py").exists()),
        None
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei dash_utils/config.py. Ajuste PROJECT_ROOT.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config = importlib.import_module("dash_utils.config")
CONSUMO_DIR = Path(config.CONSUMO_DIR)
UF = "PA"
UF_DIR = CONSUMO_DIR / f"uf={UF}"

ARQUIVOS = {
    "grupo": UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

COLUNAS = {
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "tipo_grupo", "grupo", "disciplina", "media", "n_validos",
        "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "sexo", "faixa_etaria", "disciplina", "n_participantes",
        "n_validos", "media", "std",
    ],
}

COLUNA_FILTRO = "TIPO_REFERENCIA_TERRITORIAL"
REFERENCIAS = {
    "ESC": ["ESC", "RESIDENCIA"],
    "PROVA": ["PROVA"],
}

DIAG_DIR = PROJECT_ROOT / "_diagnostico05_parquet"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_ROOT)
print("Consumo:", CONSUMO_DIR)
print("PyArrow:", pa.__version__)
print("Memory Pool:", pa.default_memory_pool().backend_name)
print("Pasta temporária:", DIAG_DIR)
for nome, caminho in ARQUIVOS.items():
    if not caminho.exists():
        raise FileNotFoundError(caminho)
    pf = pq.ParquetFile(caminho)
    print(
        f"{nome}: {caminho.stat().st_size / 1024**2:.2f} MB | "
        f"{pf.metadata.num_rows:,} linhas | "
        f"{pf.metadata.num_row_groups} Row Group(s)"
    )


Projeto: /home/akel/PycharmProjects/Projeto_ENEM/app_render
Consumo: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo
PyArrow: 26.0.1
Memory Pool: mimalloc
Pasta temporária: /home/akel/PycharmProjects/Projeto_ENEM/app_render/_diagnostico05_parquet
grupo: 8.57 MB | 337,214 linhas | 1 Row Group(s)
dsp: 10.10 MB | 451,205 linhas | 1 Row Group(s)


## 2. Inventário físico dos arquivos originais

Esta etapa registra a quantidade de linhas, Row Groups, tamanho comprimido e estatísticas da coluna de referência. As estatísticas são importantes porque o leitor Parquet pode descartar Row Groups quando seus intervalos `min`/`max` não podem conter os valores procurados.

Se o arquivo tiver um único Row Group, o filtro pode selecionar menos linhas no resultado sem conseguir descartar partes menores do arquivo por Row Group.


In [2]:
def inventariar_parquet(caminho, base, variante="original"):
    pf = pq.ParquetFile(caminho)
    linhas = []
    for i in range(pf.metadata.num_row_groups):
        rg = pf.metadata.row_group(i)
        idx = next(
            (j for j in range(rg.num_columns)
             if rg.column(j).path_in_schema == COLUNA_FILTRO),
            None
        )
        st = rg.column(idx).statistics if idx is not None else None
        linhas.append({
            "base": base,
            "variante": variante,
            "row_group": i,
            "linhas": rg.num_rows,
            "tamanho_mb": rg.total_byte_size / 1024**2,
            "min_referencia": str(st.min) if st and st.has_min_max else None,
            "max_referencia": str(st.max) if st and st.has_min_max else None,
            "null_count": st.null_count if st else None,
        })
    return linhas

inventario_original = []
for base, caminho in ARQUIVOS.items():
    inventario_original.extend(inventariar_parquet(caminho, base))

df_inventario_original = pd.DataFrame(inventario_original)
display(df_inventario_original)


,base,variante,row_group,linhas,tamanho_mb,min_referencia,max_referencia,null_count
0,grupo,original,0,337214,10.338845,ESC,RESIDENCIA,0
1,dsp,original,0,451205,12.507343,ESC,PROVA,0


## 3. Geração das alternativas físicas

As bases são lidas apenas com as colunas projetadas utilizadas pelo dashboard.

- **Ordenação:** `TIPO_REFERENCIA_TERRITORIAL`.
- **Row Groups:** 50 mil, 100 mil e 250 mil linhas.
- **Particionamento:** uma pasta por valor de `TIPO_REFERENCIA_TERRITORIAL`.

A ordenação é estável e não altera os valores nem a quantidade de registros. O particionamento mantém a coluna de referência como coluna de partição, que será reconstruída pelo leitor de dataset.

A gravação utiliza compressão Zstandard. Isso padroniza a comparação, mas significa que tamanho em disco também poderá mudar em relação ao original, que pode ter sido gravado com outra configuração.


In [3]:
ROW_GROUP_SIZES = [50_000, 100_000, 250_000]
COMPRESSION = "zstd"

def gerar_variantes(base, origem):
    destino_base = DIAG_DIR / base
    destino_base.mkdir(parents=True, exist_ok=True)

    # Leitura projetada; os arquivos originais permanecem intactos.
    tabela = pq.read_table(origem, columns=COLUNAS[base])
    if COLUNA_FILTRO not in tabela.column_names:
        raise KeyError(f"{COLUNA_FILTRO} não existe em {base}")

    # Ordena pelo campo usado no filtro.
    tabela_ordenada = tabela.sort_by([(COLUNA_FILTRO, "ascending")])

    saidas = {}
    for tamanho in ROW_GROUP_SIZES:
        nome = f"ordenado_rg_{tamanho}"
        caminho = destino_base / f"{nome}.parquet"
        pq.write_table(
            tabela_ordenada,
            caminho,
            row_group_size=tamanho,
            compression=COMPRESSION,
            use_dictionary=True,
            write_statistics=True,
        )
        saidas[nome] = caminho

    # Dataset particionado por referência territorial.
    pasta_particionada = destino_base / "particionado_referencia"
    if pasta_particionada.exists():
        shutil.rmtree(pasta_particionada)

    ds.write_dataset(
        tabela,
        base_dir=str(pasta_particionada),
        format="parquet",
        partitioning=[COLUNA_FILTRO],
        partitioning_flavor="hive",
        basename_template="parte-{i}.parquet",
        existing_data_behavior="delete_matching",
        max_rows_per_file=250_000,
        max_rows_per_group=100_000,
        file_options=ds.ParquetFileFormat().make_write_options(
            compression=COMPRESSION,
            use_dictionary=True,
            write_statistics=True,
        ),
    )
    saidas["particionado"] = pasta_particionada

    print(
        f"{base}: {tabela.num_rows:,} linhas | "
        f"{tabela.num_columns} colunas | "
        f"Arrow buffers={tabela.get_total_buffer_size()/1024**2:.1f} MB"
    )
    del tabela, tabela_ordenada
    gc.collect()
    return saidas

VARIANTES = {}
for base, origem in ARQUIVOS.items():
    VARIANTES[base] = gerar_variantes(base, origem)

print("Geração concluída.")


grupo: 337,214 linhas | 12 colunas | Arrow buffers=40.1 MB
dsp: 451,205 linhas | 10 colunas | Arrow buffers=45.1 MB
Geração concluída.


## 4. Verificação das estatísticas e do particionamento

Antes de medir memória, verificamos se as novas organizações realmente criaram grupos seletivos.

Para os arquivos ordenados, esperamos que os Row Groups tenham intervalos de referência mais estreitos. Para o dataset particionado, verificamos as pastas criadas e os fragmentos encontrados pelo PyArrow.

Não basta aumentar o número de Row Groups: se cada grupo continuar contendo todas as categorias, o filtro ainda poderá precisar ler todos eles.


In [4]:
inventario_variantes = []
for base in ["grupo", "dsp"]:
    for nome, caminho in VARIANTES[base].items():
        if nome == "particionado":
            dataset = ds.dataset(str(caminho), format="parquet", partitioning="hive")
            fragments = list(dataset.get_fragments())
            inventario_variantes.append({
                "base": base,
                "variante": nome,
                "row_groups_ou_fragmentos": len(fragments),
                "linhas": sum(f.metadata.num_rows for f in fragments),
                "intervalos": "partições por categoria",
            })
        else:
            itens = inventariar_parquet(caminho, base, nome)
            inventario_variantes.extend(itens)

df_inventario_variantes = pd.DataFrame(inventario_variantes)
display(df_inventario_variantes)

for base in ["grupo", "dsp"]:
    pasta = VARIANTES[base]["particionado"]
    print(f"\n{base} — diretórios de partição:")
    for p in sorted(pasta.iterdir()):
        if p.is_dir():
            print(" ", p.name)


,base,variante,row_group,linhas,tamanho_mb,min_referencia,max_referencia,null_count,row_groups_ou_fragmentos,intervalos
0,grupo,ordenado_rg_50000,0.0,50000,1.075858,ESC,ESC,0.0,NaN,NaN
1,grupo,ordenado_rg_50000,1.0,50000,1.293400,ESC,ESC,0.0,NaN,NaN
2,grupo,ordenado_rg_50000,2.0,50000,1.291761,ESC,ESC,0.0,NaN,NaN
3,grupo,ordenado_rg_50000,3.0,50000,1.245373,ESC,ESC,0.0,NaN,NaN
4,grupo,ordenado_rg_50000,4.0,50000,1.479774,ESC,PROVA,0.0,NaN,NaN
5,grupo,ordenado_rg_50000,5.0,50000,1.493225,PROVA,PROVA,0.0,NaN,NaN
6,grupo,ordenado_rg_50000,6.0,37214,1.098756,PROVA,RESIDENCIA,0.0,NaN,NaN
7,grupo,ordenado_rg_100000,0.0,100000,2.258011,ESC,ESC,0.0,NaN,NaN
8,grupo,ordenado_rg_100000,1.0,100000,2.343282,ESC,ESC,0.0,NaN,NaN
9,grupo,ordenado_rg_100000,2.0,100000,2.801624,ESC,PROVA,0.0,NaN,NaN



grupo — diretórios de partição:
  TIPO_REFERENCIA_TERRITORIAL=ESC
  TIPO_REFERENCIA_TERRITORIAL=PROVA
  TIPO_REFERENCIA_TERRITORIAL=RESIDENCIA

dsp — diretórios de partição:
  TIPO_REFERENCIA_TERRITORIAL=ESC
  TIPO_REFERENCIA_TERRITORIAL=PROVA


## 5. Medição isolada de memória: leitura filtrada

Cada leitura é executada em um subprocesso independente, para reduzir a influência de memória retida por testes anteriores no kernel do notebook.

São medidos:

- RSS e USS antes e depois da leitura;
- pico do Memory Pool do PyArrow;
- tempo;
- linhas e buffers da tabela resultante.

O teste utiliza o filtro ESC (`ESC` + `RESIDENCIA`) e PROVA separadamente. Para arquivos ordenados, o leitor pode aproveitar as estatísticas dos Row Groups. Para o dataset particionado, o filtro pode selecionar as partições correspondentes.

A leitura do dataset particionado utiliza `pyarrow.dataset.Scanner`. Para os arquivos Parquet individuais, também é utilizado `Scanner`, mantendo o mesmo mecanismo de leitura entre as variantes.


In [5]:
WORKER = r'''
import sys, json, gc, time
import psutil
import pyarrow as pa
import pyarrow.dataset as ds

p = json.loads(sys.argv[1])
proc = psutil.Process()
pool = pa.default_memory_pool()

def snap():
    m = proc.memory_full_info()
    return {
        "rss_mb": m.rss / 1024**2,
        "uss_mb": getattr(m, "uss", 0) / 1024**2,
        "pool_mb": pool.bytes_allocated() / 1024**2,
        "pool_max_mb": pool.max_memory() / 1024**2,
    }

def filtro(ref):
    vals = ["ESC", "RESIDENCIA"] if ref == "ESC" else ["PROVA"]
    return ds.field("TIPO_REFERENCIA_TERRITORIAL").isin(vals)

out = {
    "base": p["base"],
    "variante": p["variante"],
    "referencia": p["referencia"],
}
baseline = snap()
t0 = time.perf_counter()

dataset = ds.dataset(
    p["caminho"],
    format="parquet",
    partitioning="hive" if p["particionado"] else None,
)
scanner = dataset.scanner(
    columns=p["columns"],
    filter=filtro(p["referencia"]),
    use_threads=p["use_threads"],
)
table = scanner.to_table()
t1 = time.perf_counter()

final = snap()
out.update({
    "baseline": baseline,
    "final": final,
    "tempo_s": t1 - t0,
    "linhas": table.num_rows,
    "buffers_mb": table.get_total_buffer_size() / 1024**2,
    "nbytes_mb": table.nbytes / 1024**2,
    "fragmentos": len(list(dataset.get_fragments(filter=filtro(p["referencia"])))),
})
del table, scanner, dataset
gc.collect()
out["apos_gc"] = snap()
print(json.dumps(out))
'''

def executar_leitura(base, variante, caminho, referencia, particionado=False, use_threads=True):
    payload = {
        "base": base,
        "variante": variante,
        "caminho": str(caminho),
        "particionado": particionado,
        "referencia": referencia,
        "columns": COLUNAS[base],
        "use_threads": use_threads,
    }
    proc = subprocess.run(
        [sys.executable, "-c", WORKER, json.dumps(payload)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"Falha em {base}/{variante}/{referencia}\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    linhas_json = [x for x in proc.stdout.splitlines() if x.strip().startswith("{")]
    if not linhas_json:
        raise RuntimeError(f"Worker sem JSON: {proc.stdout}")
    return json.loads(linhas_json[-1])


## 6. Execução dos cenários

A comparação inclui o arquivo original e todas as alternativas geradas. O cenário original funciona como referência.

Se o tempo de execução for elevado, execute primeiro apenas uma base e uma referência, ajustando as listas abaixo. Os resultados de cada execução são armazenados em `resultados`.


In [6]:
CENARIOS = []
for base in ["grupo", "dsp"]:
    CENARIOS.append((base, "original", ARQUIVOS[base], False))
    for nome, caminho in VARIANTES[base].items():
        CENARIOS.append((base, nome, caminho, nome == "particionado"))

resultados = []
for base, nome, caminho, particionado in CENARIOS:
    for referencia in ["ESC", "PROVA"]:
        r = executar_leitura(
            base, nome, caminho, referencia,
            particionado=particionado,
            use_threads=True,
        )
        resultados.append(r)
        print(
            f"{base:5s} | {nome:22s} | {referencia:5s} | "
            f"linhas={r['linhas']:>8,} | "
            f"USS Δ={r['final']['uss_mb']-r['baseline']['uss_mb']:7.1f} MB | "
            f"pool pico={r['final']['pool_max_mb']:6.1f} MB | "
            f"tempo={r['tempo_s']:.3f} s"
        )


grupo | original               | ESC   | linhas= 205,400 | USS Δ=  157.0 MB | pool pico=  71.2 MB | tempo=0.037 s
grupo | original               | PROVA | linhas= 131,814 | USS Δ=  163.1 MB | pool pico=  62.6 MB | tempo=0.034 s
grupo | ordenado_rg_50000      | ESC   | linhas= 205,400 | USS Δ=  175.6 MB | pool pico=  51.7 MB | tempo=0.035 s
grupo | ordenado_rg_50000      | PROVA | linhas= 131,814 | USS Δ=  141.3 MB | pool pico=  27.6 MB | tempo=0.028 s
grupo | ordenado_rg_100000     | ESC   | linhas= 205,400 | USS Δ=  145.6 MB | pool pico=  53.7 MB | tempo=0.035 s
grupo | ordenado_rg_100000     | PROVA | linhas= 131,814 | USS Δ=  123.2 MB | pool pico=  31.2 MB | tempo=0.029 s
grupo | ordenado_rg_250000     | ESC   | linhas= 205,400 | USS Δ=  147.2 MB | pool pico=  61.0 MB | tempo=0.039 s
grupo | ordenado_rg_250000     | PROVA | linhas= 131,814 | USS Δ=  153.2 MB | pool pico=  62.2 MB | tempo=0.036 s
grupo | particionado           | ESC   | linhas= 205,400 | USS Δ=  123.9 MB | pool pico=

## 7. Tabela comparativa

O `delta_uss_mb` é a diferença entre USS após a leitura e USS no início do subprocesso. `pool_max_mb` é o pico registrado pelo Memory Pool durante o processo. Eles são indicadores diferentes e não devem ser interpretados como medidas equivalentes.

Compare cada variante com o original da mesma base e referência. A quantidade de linhas deve ser igual entre as estratégias para uma mesma combinação de base e referência.


In [7]:
def resumir_resultado(r):
    return {
        "base": r["base"],
        "variante": r["variante"],
        "referencia": r["referencia"],
        "linhas": r["linhas"],
        "tempo_s": r["tempo_s"],
        "delta_uss_mb": r["final"]["uss_mb"] - r["baseline"]["uss_mb"],
        "delta_rss_mb": r["final"]["rss_mb"] - r["baseline"]["rss_mb"],
        "pool_final_mb": r["final"]["pool_mb"],
        "pool_pico_mb": r["final"]["pool_max_mb"],
        "buffers_mb": r["buffers_mb"],
        "fragmentos_selecionados": r["fragmentos"],
    }

df_resultados = pd.DataFrame([resumir_resultado(r) for r in resultados])
display(
    df_resultados.sort_values(["base", "referencia", "delta_uss_mb"])
    .reset_index(drop=True)
)


,base,variante,referencia,linhas,tempo_s,delta_uss_mb,delta_rss_mb,pool_final_mb,pool_pico_mb,buffers_mb,fragmentos_selecionados
0,dsp,particionado,ESC,263761,0.026390,141.964844,148.226562,25.218445,28.213684,25.208113,2
1,dsp,ordenado_rg_100000,ESC,263761,0.034435,145.179688,151.441406,26.347290,54.473022,26.344007,1
2,dsp,ordenado_rg_250000,ESC,263761,0.038373,149.085938,155.347656,26.347168,59.195496,26.344046,1
3,dsp,original,ESC,263761,0.041636,160.988281,167.238281,26.347717,77.283386,26.344055,1
4,dsp,ordenado_rg_50000,ESC,263761,0.035856,161.507812,167.832031,26.348755,52.830505,26.344122,1
5,dsp,ordenado_rg_250000,PROVA,187444,0.026588,102.855469,109.101562,19.005432,42.629517,19.003639,1
6,dsp,particionado,PROVA,187444,0.029957,123.769531,130.023438,18.185425,21.427673,18.177263,1
7,dsp,ordenado_rg_50000,PROVA,187444,0.027536,133.292969,139.640625,19.007385,40.656860,19.003754,1
8,dsp,ordenado_rg_100000,PROVA,187444,0.036142,137.164062,143.449219,19.006653,38.997620,19.003677,1
9,dsp,original,PROVA,187444,0.039085,168.648438,174.953125,19.007202,69.942871,19.003725,1


## 8. Comparação relativa ao arquivo original

Esta tabela calcula a diferença percentual de USS e tempo em relação ao original da mesma base e referência. Valores negativos indicam redução; valores positivos indicam aumento.

A comparação percentual só deve ser considerada quando a quantidade de linhas resultantes for idêntica.


In [8]:
originais = (
    df_resultados[df_resultados["variante"] == "original"]
    [["base", "referencia", "linhas", "delta_uss_mb", "tempo_s", "pool_pico_mb"]]
    .rename(columns={
        "delta_uss_mb": "uss_original_mb",
        "tempo_s": "tempo_original_s",
        "pool_pico_mb": "pool_original_mb",
    })
)

df_relativo = df_resultados.merge(
    originais,
    on=["base", "referencia", "linhas"],
    how="left",
)
df_relativo["variacao_uss_pct"] = (
    (df_relativo["delta_uss_mb"] / df_relativo["uss_original_mb"]) - 1
) * 100
df_relativo["variacao_tempo_pct"] = (
    (df_relativo["tempo_s"] / df_relativo["tempo_original_s"]) - 1
) * 100
df_relativo["variacao_pool_pct"] = (
    (df_relativo["pool_pico_mb"] / df_relativo["pool_original_mb"]) - 1
) * 100

display(
    df_relativo[
        [
            "base", "variante", "referencia", "linhas",
            "delta_uss_mb", "variacao_uss_pct",
            "tempo_s", "variacao_tempo_pct",
            "pool_pico_mb", "variacao_pool_pct",
        ]
    ].sort_values(["base", "referencia", "delta_uss_mb"])
)


,base,variante,referencia,linhas,delta_uss_mb,variacao_uss_pct,tempo_s,variacao_tempo_pct,pool_pico_mb,variacao_pool_pct
18,dsp,particionado,ESC,263761,141.964844,-11.816660,0.026390,-36.617731,28.213684,-63.493209
14,dsp,ordenado_rg_100000,ESC,263761,145.179688,-9.819717,0.034435,-17.294395,54.473022,-29.515223
16,dsp,ordenado_rg_250000,ESC,263761,149.085938,-7.393298,0.038373,-7.836113,59.195496,-23.404630
10,dsp,original,ESC,263761,160.988281,0.000000,0.041636,0.000000,77.283386,0.000000
12,dsp,ordenado_rg_50000,ESC,263761,161.507812,0.322714,0.035856,-13.881280,52.830505,-31.640540
17,dsp,ordenado_rg_250000,PROVA,187444,102.855469,-39.011905,0.026588,-31.974767,42.629517,-39.050948
19,dsp,particionado,PROVA,187444,123.769531,-26.610923,0.029957,-23.354320,21.427673,-69.364035
13,dsp,ordenado_rg_50000,PROVA,187444,133.292969,-20.964006,0.027536,-29.547810,40.656860,-41.871331
15,dsp,ordenado_rg_100000,PROVA,187444,137.164062,-18.668643,0.036142,-7.530094,38.997620,-44.243611
11,dsp,original,PROVA,187444,168.648438,0.000000,0.039085,0.000000,69.942871,0.000000


## 9. Validação de equivalência dos resultados

Além da quantidade de linhas, precisamos confirmar que as leituras filtradas retornam os mesmos registros. A comparação abaixo faz uma validação por contagem e soma de hashes das linhas, sem depender da ordem física dos registros.

A validação lê cada variante novamente. É uma etapa adicional de I/O e memória; execute-a depois de salvar os resultados de desempenho.


In [9]:
def ler_para_validacao(base, caminho, referencia, particionado=False):
    dataset = ds.dataset(
        str(caminho),
        format="parquet",
        partitioning="hive" if particionado else None,
    )
    vals = REFERENCIAS[referencia]
    scanner = dataset.scanner(
        columns=COLUNAS[base],
        filter=ds.field(COLUNA_FILTRO).isin(vals),
        use_threads=True,
    )
    return scanner.to_table().to_pandas()

def assinatura_dataframe(df):
    # Hash de cada linha, independentemente da ordem.
    hashes = pd.util.hash_pandas_object(
        df, index=False
    ).to_numpy(dtype=np.uint64)

    return {
        "linhas": len(df),
        "hash_soma_uint64": int(hashes.sum(dtype=np.uint64)),
        "hash_xor_uint64": int(np.bitwise_xor.reduce(hashes, initial=0)),
    }

validacoes = []
for base, nome, caminho, particionado in CENARIOS:
    for referencia in ["ESC", "PROVA"]:
        df = ler_para_validacao(base, caminho, referencia, particionado)
        assinatura = assinatura_dataframe(df)
        validacoes.append({
            "base": base,
            "variante": nome,
            "referencia": referencia,
            **assinatura,
        })
        del df
        gc.collect()

df_validacoes = pd.DataFrame(validacoes)
display(df_validacoes.sort_values(["base", "referencia", "variante"]))


,base,variante,referencia,linhas,hash_soma_uint64,hash_xor_uint64
14,dsp,ordenado_rg_100000,ESC,263761,4508641949741006091,10981785790140965475
16,dsp,ordenado_rg_250000,ESC,263761,4508641949741006091,10981785790140965475
12,dsp,ordenado_rg_50000,ESC,263761,4508641949741006091,10981785790140965475
10,dsp,original,ESC,263761,4508641949741006091,10981785790140965475
18,dsp,particionado,ESC,263761,4508641949741006091,10981785790140965475
15,dsp,ordenado_rg_100000,PROVA,187444,969243322286495189,7290504266962405319
17,dsp,ordenado_rg_250000,PROVA,187444,969243322286495189,7290504266962405319
13,dsp,ordenado_rg_50000,PROVA,187444,969243322286495189,7290504266962405319
11,dsp,original,PROVA,187444,969243322286495189,7290504266962405319
19,dsp,particionado,PROVA,187444,969243322286495189,7290504266962405319


## 10. Interpretação e decisão

Use os resultados para responder às perguntas:

1. **Os Row Groups ordenados ficaram seletivos?** Verifique se `min_referencia` e `max_referencia` ficam restritos a poucas categorias.
2. **O filtro selecionou menos fragmentos?** Observe `fragmentos_selecionados`, especialmente no dataset particionado.
3. **Houve redução de USS e do pico do Memory Pool?** Compare com o original da mesma base e referência.
4. **O ganho se manteve nas duas bases e nas duas referências?** Um resultado isolado não basta para alterar o pipeline.
5. **Os resultados são equivalentes?** A validação por contagem e hashes deve coincidir com o original.

### Como interpretar possíveis resultados

- **Menos Row Groups lidos, mas USS semelhante:** a leitura pode já ser dominada por alocações temporárias, buffers ou comportamento do alocador. A organização física, nesse caso, não resolve o principal custo.
- **Redução de USS e tempo:** há evidência de que ordenação/particionamento pode ajudar. Ainda será necessário testar o carregamento integrado do dashboard.
- **Redução de tempo, mas não de memória:** pode ser uma otimização útil, mas não resolve o limite de RAM do Render.
- **Nenhuma redução consistente:** é plausível que o custo seja inerente à representação e ao volume das colunas necessárias. Nesse caso, devemos investigar outras frentes, como leitura por etapas, cache, ciclo de vida dos DataFrames e arquitetura de carregamento.

**Não altere ainda os Parquets de produção.** A decisão depende da equivalência dos dados e da repetibilidade dos ganhos. O diagnóstico não presume que a organização física necessariamente reduzirá o consumo.


## 11. Limpeza opcional dos arquivos temporários

Execute somente depois de concluir a análise e, se necessário, salvar os DataFrames de resultados. Esta célula remove exclusivamente a pasta `_diagnostico05_parquet` criada por este notebook.


In [10]:
# ATENÇÃO: descomente apenas se quiser apagar os arquivos temporários gerados.
# shutil.rmtree(DIAG_DIR)
# print("Pasta temporária removida:", DIAG_DIR)
